#### Decorators
Decorators are a powerful and flexible feature in Python that allows you to modify the behavior of a function or class method. They are commonly used to add functionality to functions or methods without modifying their actual code. This lesson covers the basics of decorators, including how to create and use them.

In [1]:
### function copy
### closures
### decorators

In [2]:
## function copy
def welcome():
    return "Welcome to the advanced python course"

welcome()

'Welcome to the advanced python course'

In [3]:
wel=welcome
print(wel())
del welcome
print(wel())

Welcome to the advanced python course
Welcome to the advanced python course


In [4]:
##closures functions

def main_welcome(msg):
   
    def sub_welcome_method():
        print("Welcome to the advance python course")
        print(msg)
        print("Please learn these concepts properly")
    return sub_welcome_method()

In [5]:
main_welcome("Welcome everyone")

Welcome to the advance python course
Welcome everyone
Please learn these concepts properly


In [6]:
def main_welcome(func):
   
    def sub_welcome_method():
        print("Welcome to the advance python course")
        func("Welcome everyone to this tutorial")
        print("Please learn these concepts properly")
    return sub_welcome_method()

In [7]:
main_welcome(print)

Welcome to the advance python course
Welcome everyone to this tutorial
Please learn these concepts properly


In [8]:
def main_welcome(func,lst):
   
    def sub_welcome_method():
        print("Welcome to the advance python course")
        print(func(lst))
        print("Please learn these concepts properly")
    return sub_welcome_method()

In [9]:
main_welcome(len,[1,2,3,4,5])

Welcome to the advance python course
5
Please learn these concepts properly


In [10]:
len([1,2,3,4,5,6])

6

In [11]:
### Decorator
def main_welcome(func):
   
    def sub_welcome_method():
        print("Welcome to the advance python course")
        func()
        print("Please learn these concepts properly")
    return sub_welcome_method()

In [12]:
def coure_introduction():
    print("This is an advanced python course")

coure_introduction()

This is an advanced python course


In [13]:
main_welcome(coure_introduction)

Welcome to the advance python course
This is an advanced python course
Please learn these concepts properly


In [14]:
@main_welcome
def coure_introduction():
    print("This is an advanced python course")

Welcome to the advance python course
This is an advanced python course
Please learn these concepts properly


In [15]:
## Decorator

def my_decorator(func):
    def wrapper():
        print("Something is happening before the function is called.")
        func()
        print("Something is happening after the function is called.")
    return wrapper

In [16]:
@my_decorator
def say_hello():
    print("Hello!")



In [17]:
say_hello()

Something is happening before the function is called.
Hello!
Something is happening after the function is called.


In [18]:
## Decorators WWith arguments
def repeat(n):
    def decorator(func):
        def wrapper(*args, **kwargs):
            for _ in range(n):
                func(*args, **kwargs)
        return wrapper
    return decorator

In [19]:
@repeat(3)
def say_hello():
    print("Hello")

In [20]:
say_hello()

Hello
Hello
Hello


#### Conclusion
Decorators are a powerful tool in Python for extending and modifying the behavior of functions and methods. They provide a clean and readable way to add functionality such as logging, timing, access control, and more without changing the original code. Understanding and using decorators effectively can significantly enhance your Python programming skills.

---
# 🎯 Decorators: Interview Mastery & Production-Grade Patterns
---
This section covers advanced decorator patterns, metadata preservation with `functools.wraps`, class-based decorators, decorator factories with arguments, stacking order, and industry-standard decorators (timers, memoization, retries).

### Key Interview Topics:
1. **Preserving Metadata with `functools.wraps` (The #1 Interview Trap)**
2. **Decorating Functions with Arguments & Return Values (`*args, **kwargs`)**
3. **Decorator Factories (Decorators Taking Arguments)**
4. **Class-Based Decorators (`__call__` and State Maintenance)**
5. **Stacking Multiple Decorators & Execution Order**
6. **Production Decorators: `@timer`, `@memoize`, `@retry`, `@validate_types`**

### 1. Preserving Metadata with `functools.wraps` (The Classic Interview Trap)
**Interview Question:** *What happens to a decorated function's `__name__` and `__doc__` if you do not use `@functools.wraps`? Why does this break production code?*

- Without `@wraps`, the decorated function takes on the name and docstring of the inner `wrapper` function (`'wrapper'`).
- This breaks debuggers, logging frameworks, documentation generators (Sphinx), and `inspect` introspection!

In [21]:
import functools

# ❌ BAD Decorator: Omits @functools.wraps
def bad_decorator(func):
    def wrapper(*args, **kwargs):
        """Bad wrapper docstring."""
        return func(*args, **kwargs)
    return wrapper

@bad_decorator
def calculate_tax(amount):
    """Calculates sales tax."""
    return amount * 0.08

print("Without @wraps:")
print("Function Name:", calculate_tax.__name__)  # Prints 'wrapper'!
print("Docstring:    ", calculate_tax.__doc__)   # Prints 'Bad wrapper docstring.'

# ✅ GOOD Decorator: Uses @functools.wraps
def good_decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@good_decorator
def calculate_discount(amount):
    """Calculates customer discount."""
    return amount * 0.15

print("\nWith @functools.wraps:")
print("Function Name:", calculate_discount.__name__)  # Correctly prints 'calculate_discount'
print("Docstring:    ", calculate_discount.__doc__)   # Correctly prints 'Calculates customer discount.'

Without @wraps:
Function Name: wrapper
Docstring:     Bad wrapper docstring.

With @functools.wraps:
Function Name: calculate_discount
Docstring:     Calculates customer discount.


### 2. Execution Order of Multiple (Stacked) Decorators
**Interview Question:** *Given `@decorator_one` over `@decorator_two`, in what order are the decorators applied and executed?*

- **Application order (wrapping):** Bottom-up (`decorator_two` wraps first, then `decorator_one`).
  Equivalent to: `func = decorator_one(decorator_two(func))`
- **Execution order (runtime):** Top-down (`decorator_one` wrapper runs first, then delegates to `decorator_two` wrapper).

In [22]:
# Stacking Decorators Demonstration
def bold(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return "<b>" + func(*args, **kwargs) + "</b>"
    return wrapper

def italic(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return "<i>" + func(*args, **kwargs) + "</i>"
    return wrapper

# Applied bottom-up: italic first, then bold!
@bold
@italic
def format_text(text):
    return text

print("Stacked Result:", format_text("Hello World"))

Stacked Result: <b><i>Hello World</i></b>


### 3. Class-Based Decorators (`__call__`) & State Maintenance
**Interview Question:** *How do you write a decorator using a Python class? What is the main advantage of class decorators over function decorators?*

- A class implementing `__call__(self, *args, **kwargs)` can act as a decorator.
- **Advantage:** Easy state retention (e.g. keeping track of how many times a function has been called, caching state, or rate limiting).

In [23]:
# Class-Based Decorator: Call Counter
class CallCounter:
    def __init__(self, func):
        self.func = func
        self.count = 0
        functools.wraps(func)(self)
        
    def __call__(self, *args, **kwargs):
        self.count += 1
        print(f"Function '{self.func.__name__}' called {self.count} time(s).")
        return self.func(*args, **kwargs)

@CallCounter
def process_order(order_id):
    return f"Order #{order_id} processed"

process_order(101)
process_order(102)
print("Total Invocation Count:", process_order.count)

Function 'process_order' called 1 time(s).
Function 'process_order' called 2 time(s).
Total Invocation Count: 2


### 4. Production Pattern 1: Performance Timer Decorator (`@timer`)
**Interview Question:** *Write a production-ready timing decorator using `time.perf_counter()`.*

In [24]:
import time

def timer(func):
    """Times the execution of a function."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        duration = time.perf_counter() - start_time
        print(f"⏱️ [{func.__name__}] finished in {duration:.6f} seconds.")
        return result
    return wrapper

@timer
def compute_heavy_task(n):
    return sum(i ** 2 for i in range(n))

compute_heavy_task(1_000_000)

⏱️ [compute_heavy_task] finished in 0.111491 seconds.


333332833333500000

### 5. Production Pattern 2: Memoization / Caching Decorator (`@memoize`)
**Interview Question:** *Implement your own `@memoize` caching decorator from scratch to optimize exponential recursive algorithms.*

In [25]:
def memoize(func):
    """Custom in-memory LRU-style memoization decorator."""
    cache = {}
    
    @functools.wraps(func)
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]
        
    wrapper.cache = cache # Expose cache for inspection
    return wrapper

@memoize
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

print("Fibonacci(35):", fibonacci(35))
print("Total cached calls:", len(fibonacci.cache))

Fibonacci(35): 9227465
Total cached calls: 36


### 6. Production Pattern 3: Exponential Backoff Retry Decorator (`@retry`)
**Interview Question:** *Write a parameterized decorator `@retry(max_retries, delay, backoff)` for robust API requests.*

In [26]:
def retry(max_retries=3, delay=0.1, backoff=2.0):
    """Decorator factory that retries flaky operations with exponential backoff."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            current_delay = delay
            for attempt in range(1, max_retries + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if attempt == max_retries:
                        print(f"❌ Attempt {attempt} failed. Max retries reached!")
                        raise e
                    print(f"⚠️ Attempt {attempt} failed ({e}). Retrying in {current_delay:.2f}s...")
                    time.sleep(current_delay)
                    current_delay *= backoff
        return wrapper
    return decorator

# Test retry on simulated unstable network call:
attempts_made = 0
@retry(max_retries=3, delay=0.05, backoff=2.0)
def fetch_api_data():
    global attempts_made
    attempts_made += 1
    if attempts_made < 3:
        raise ConnectionError("Network timeout")
    return {"status": 200, "data": "Success"}

print("API Result:", fetch_api_data())

⚠️ Attempt 1 failed (Network timeout). Retrying in 0.05s...


⚠️ Attempt 2 failed (Network timeout). Retrying in 0.10s...


API Result: {'status': 200, 'data': 'Success'}
